# Low Memory Improved RNN Model with CPU Optimization

This notebook implements the improved Bidirectional LSTM model with optimizations to utilize all CPU cores while minimizing memory usage.

## Memory Usage Warning

⚠️ **IMPORTANT**: This notebook is optimized for systems with limited RAM. If you're experiencing memory issues:

1. Close other applications before running this notebook
2. Run cells one at a time and monitor memory usage
3. If you still experience issues, try reducing `batch_size` further
4. Consider using a smaller subset of the data for training

## CPU Optimization Setup

First, we'll configure TensorFlow to use all available CPU cores. This must be done before importing TensorFlow.

In [ ]:
import os
import numpy as np
import multiprocessing
import gc  # Garbage collector for memory management

# Get the number of available CPU cores
num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores available: {num_cores}")

# Set TensorFlow to use all available cores
os.environ["TF_NUM_INTRAOP_THREADS"] = str(num_cores)
os.environ["TF_NUM_INTEROP_THREADS"] = str(num_cores)
os.environ["OMP_NUM_THREADS"] = str(num_cores)
os.environ["MKL_NUM_THREADS"] = str(num_cores)

# Memory optimization for TensorFlow
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Only for GPU, but doesn't hurt
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce logging to save memory

# Now import TensorFlow
import tensorflow as tf

# Configure TensorFlow session
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)
tf.config.set_soft_device_placement(True)

# Memory growth - limit memory usage
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

print(f"TensorFlow is configured to use {num_cores} CPU cores with memory optimizations")

In [ ]:
# Import remaining libraries
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import re
import requests
import zipfile
import io

## Parameters

We'll use a smaller batch size to reduce memory usage.

In [ ]:
# Parameters
max_features = 10000  # Vocabulary size
max_len = 500  # Maximum sequence length
embedding_dim = 100  # Dimension of GloVe embeddings

# Use a smaller batch size to reduce memory usage
# If you're having memory issues, reduce batch_size further (to 16 or even 8)
batch_size = 32  # Smaller batch size uses less memory
print(f"Using batch size: {batch_size} to minimize memory usage")

epochs = 10  # Reduced epochs to save time and memory

## Load and Preprocess the IMDB Dataset

We'll use a memory-efficient approach to data loading.

In [ ]:
# Load the IMDB dataset
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

# Pad sequences to ensure uniform input size
X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)

# Get word index for later use
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

In [ ]:
# Create memory-efficient TensorFlow datasets
AUTOTUNE = tf.data.experimental.AUTOTUNE

# Convert to TensorFlow Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Configure for performance WITHOUT caching to save memory
train_dataset = train_dataset.shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)  # No .cache() to save memory
test_dataset = test_dataset.batch(batch_size).prefetch(AUTOTUNE)

# Create validation dataset
val_size = int(0.2 * len(X_train))
train_dataset_final = train_dataset.skip(val_size)
val_dataset = train_dataset.take(val_size)

print("TensorFlow datasets created with memory-efficient settings")

# Force garbage collection to free memory
gc.collect()

## Download and Load GloVe Embeddings (Optional)

This section can be skipped if you're very low on memory.

In [ ]:
# MEMORY USAGE WARNING: If you're very low on RAM, set use_glove = False and skip the next two cells
# Set use_glove = False to skip using pre-trained embeddings
use_glove = True  # Set to False if you're very low on memory

if not use_glove:
    print("Skipping GloVe embeddings to save memory")

In [ ]:
# Function to download GloVe embeddings if not already downloaded
def download_glove_embeddings():
    if not use_glove:
        return None
        
    glove_dir = 'glove'
    if not os.path.exists(glove_dir):
        os.makedirs(glove_dir)
    
    glove_path = os.path.join(glove_dir, 'glove.6B.100d.txt')
    if not os.path.exists(glove_path):
        print("Downloading GloVe embeddings...")
        url = "https://nlp.stanford.edu/data/glove.6B.zip"
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall(glove_dir)
        print("Download complete!")
    else:
        print("GloVe embeddings already downloaded.")
    
    return glove_path

# Only run if use_glove is True
if use_glove:
    glove_path = download_glove_embeddings()

In [ ]:
# Load GloVe embeddings with memory optimization
def load_glove_embeddings(glove_path):
    if not use_glove or glove_path is None:
        return None
        
    print("Loading GloVe embeddings...")
    embeddings_index = {}
    
    # Process file in chunks to save memory
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in word_index and word_index[word] < max_features:
                coefs = np.asarray(values[1:], dtype='float32')
                embeddings_index[word] = coefs
    
    print(f"Found {len(embeddings_index)} relevant word vectors.")
    
    # Create embedding matrix
    embedding_matrix = np.zeros((max_features, embedding_dim))
    for word, i in word_index.items():
        if i < max_features:
            embedding_vector = embeddings_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
    
    # Clear memory
    del embeddings_index
    gc.collect()
    
    return embedding_matrix

# Only run if use_glove is True
embedding_matrix = None
if use_glove and 'glove_path' in locals():
    embedding_matrix = load_glove_embeddings(glove_path)

## Build the Improved Model

In [ ]:
# Build an improved model with LSTM, Bidirectional, and Dropout
print("Building model...")
model = Sequential()

# Add embedding layer (with or without pre-trained embeddings)
if use_glove and embedding_matrix is not None:
    model.add(Embedding(max_features, embedding_dim, 
                       weights=[embedding_matrix],
                       input_length=max_len,
                       trainable=False))  # Freeze the embeddings initially
    # Clear memory
    del embedding_matrix
    gc.collect()
else:
    model.add(Embedding(max_features, embedding_dim, input_length=max_len))

# Use a simpler architecture to save memory
model.add(Bidirectional(LSTM(32, return_sequences=False)))  # Reduced units, no return_sequences
model.add(Dropout(0.3))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model with a lower learning rate
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Display model summary
model.summary()

## Train the Model with Callbacks

In [ ]:
# Set up callbacks for early stopping and learning rate reduction
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,  # Reduced patience to save time
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,  # Reduced patience to save time
    min_lr=0.0001
)

In [ ]:
# Train the model using the optimized tf.data datasets
print("Training model with optimized CPU utilization and memory efficiency...")
history = model.fit(
    train_dataset_final,
    epochs=epochs,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr],
    verbose=2  # Use less verbose output to save memory
)

## Evaluate and Save the Model

In [ ]:
# Evaluate the model
print("Evaluating model...")
loss, accuracy = model.evaluate(test_dataset, verbose=2)
print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
# Save the model
model.save('improved_lstm_imdb_low_memory.h5')
print("Model saved as 'improved_lstm_imdb_low_memory.h5'")

## Test the Model on Sample Reviews

In [ ]:
# Function for text preprocessing and prediction
def preprocess_text(text):
    # Clean the text
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    
    # Tokenize and convert to sequence
    words = text.split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]  # Unknown words are mapped to 2
    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)
    
    return padded_review

def predict_sentiment(text):
    preprocessed_text = preprocess_text(text)
    prediction = model.predict(preprocessed_text, verbose=0)[0][0]
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    return sentiment, prediction

In [ ]:
# Example usage
sample_reviews = [
    "This movie was fantastic! The acting was superb and the plot was engaging.",
    "Terrible film. Complete waste of time and money. The acting was wooden.",
    "I'm not sure how I feel about this movie. It had good and bad moments."
]

print("\nTesting with sample reviews:")
for review in sample_reviews:
    sentiment, score = predict_sentiment(review)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}")
    print(f"Score: {score:.4f}")
    print("-" * 50)

## Visualize Training History (Optional)

Skip this cell if you're very low on memory.

In [ ]:
# MEMORY USAGE WARNING: Skip this cell if you're very low on memory
# Skip the visualization cell at the end
# Plot training history
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='lower right')
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('training_history_low_memory.png')
    print("Training history plot saved as 'training_history_low_memory.png'")
except Exception as e:
    print(f"Could not generate plot: {e}")

## Memory and CPU Optimization Summary

This notebook implements several optimizations to maximize CPU utilization while minimizing memory usage:

1. **Memory-efficient data loading**: No caching, selective loading of embeddings
2. **Simplified model architecture**: Single Bidirectional LSTM layer instead of stacked layers
3. **Garbage collection**: Explicit memory cleanup at key points
4. **Smaller batch size**: Reduces memory footprint during training
5. **Optional GloVe embeddings**: Can be disabled to save memory
6. **Multi-threading configuration**: Sets TensorFlow to use all available CPU cores

These optimizations should allow the model to train on systems with limited RAM while still utilizing all CPU cores.